# Task 2 — Exploratory Data Analysis (EDA)
This notebook explores the Titanic training dataset with Pandas, NumPy, Matplotlib, Seaborn and Plotly. It covers data quality, summary statistics, distributions, outliers, correlations and feature relationships.


In [ ]:
%pip -q install pandas numpy matplotlib seaborn plotly tabulate


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display

DATA_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / 'titanic.csv'
if not DATA_PATH.exists():
    with urlopen(DATA_URL, timeout=30) as response:
        DATA_PATH.write_bytes(response.read())
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())


## 1. Dataset structure and data types


In [ ]:
df.info()
display(df.describe(include='all').T)


## 2. Missing-value analysis


In [ ]:
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_percent': (df.isna().mean()*100).round(2)})
display(missing.sort_values('missing_percent', ascending=False))
plt.figure(figsize=(11,4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title('Missing-value pattern')
plt.show()


## 3. Summary statistics, median and skewness


In [ ]:
numeric = df.select_dtypes(include=np.number)
summary = numeric.describe().T
summary['median'] = numeric.median()
summary['skewness'] = numeric.skew()
display(summary.round(3))


## 4. Histograms and boxplots


In [ ]:
features = ['Age', 'SibSp', 'Parch', 'Fare', 'Pclass']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flat, features):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
for ax in axes.flat[len(features):]: ax.remove()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(features), figsize=(17, 4))
for ax, col in zip(axes, features):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(f'Boxplot: {col}')
plt.tight_layout(); plt.show()


## 5. Potential outliers using the 1.5 × IQR rule


In [ ]:
outlier_rows = []
for col in features:
    values = df[col].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = int(((values < lower) | (values > upper)).sum())
    outlier_rows.append({'feature':col, 'lower_fence':lower, 'upper_fence':upper, 'potential_outliers':count, 'outlier_percent':round(100*count/len(values),2)})
outliers = pd.DataFrame(outlier_rows)
display(outliers)
print('IQR flags are candidates for investigation, not automatically data errors.')


## 6. Correlation analysis


In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.show()


## 7. Feature relationships and survival comparisons


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.barplot(data=df, x='Sex', y='Survived', hue='Sex', errorbar=None, ax=axes[0], legend=False)
axes[0].set_title('Survival rate by sex'); axes[0].set_ylabel('Survival rate')
sns.barplot(data=df, x='Pclass', y='Survived', hue='Pclass', errorbar=None, ax=axes[1], legend=False)
axes[1].set_title('Survival rate by passenger class'); axes[1].set_ylabel('Survival rate')
plt.tight_layout(); plt.show()

fig = px.scatter(df, x='Age', y='Fare', color='Survived', hover_data=['Pclass','Sex'], title='Age vs Fare by survival')
fig.show()


## 8. Simple feature engineering


In [ ]:
eda_df = df.copy()
eda_df['FamilySize'] = eda_df['SibSp'].fillna(0) + eda_df['Parch'].fillna(0) + 1
eda_df['IsAlone'] = (eda_df['FamilySize'] == 1).astype(int)
eda_df['AgeGroup'] = pd.cut(eda_df['Age'], bins=[0,12,18,35,60,np.inf], labels=['Child','Teen','Young adult','Adult','Senior'], include_lowest=True)
display(eda_df[['Age','SibSp','Parch','FamilySize','IsAlone','AgeGroup']].head(10))
display(eda_df.groupby('AgeGroup', observed=False)['Survived'].mean().rename('survival_rate'))


## 9. Findings and limitations
Use the tables and plots above to describe actual patterns. The dataset is observational: associations do not establish causation. Missingness and historical sampling can bias results. Treat IQR outliers as review candidates rather than deleting them automatically. Do not use the target column `Survived` as a predictor when training a survival model.


In [ ]:
print(f'Final dataset dimensions: {df.shape[0]} rows × {df.shape[1]} columns')
print('Duplicate rows:', df.duplicated().sum())
print('Analysis complete. Inspect each chart and table before writing conclusions.')
